In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("📊 GLOBAL INFLATION ANALYSIS - DATA PROCESSING")
print("=" * 60)

# ============================================
# 1. Load and Process Each CSV File
# ============================================

def load_wb_csv(filepath, indicator_name):
    """
    Load World Bank CSV with proper handling:
    - Data starts at row 4 (skip first 4 rows)
    - Columns: Country Name, Country Code, Indicator Name, Indicator Code, then years
    """
    # Read CSV, skip first 4 rows (metadata)
    df = pd.read_csv(filepath, skiprows=4)
    
    # Print info about loaded data
    print(f"\n📄 Loading {indicator_name}...")
    print(f"   Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"   Columns: {df.columns[:5].tolist()}...")
    
    # Keep only needed columns: Country Name, Country Code, and years 1990-2023
    years = [str(y) for y in range(1990, 2024)]
    keep_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'] + years
    keep_cols = [col for col in keep_cols if col in df.columns]
    
    df = df[keep_cols]
    
    # Remove rows where Country Name is missing (usually aggregates/regions)
    df = df.dropna(subset=['Country Name'])
    
    # Clean: remove any rows that are clearly aggregates (like "World", "East Asia", etc.)
    # Keep only actual countries (based on having a valid Country Code)
    df = df[df['Country Code'].notna()]
    df = df[df['Country Code'].str.len() == 3]  # Standard country codes are 3 letters
    
    # Melt years into rows
    melted = df.melt(
        id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
        value_vars=years,
        var_name='Year',
        value_name=indicator_name
    )
    
    # Clean up
    melted['Year'] = pd.to_numeric(melted['Year'])
    melted = melted.dropna(subset=[indicator_name])
    
    # Rename columns for consistency
    melted = melted.rename(columns={
        'Country Name': 'Country',
        'Country Code': 'economy'
    })
    
    print(f"   ✅ {len(melted)} observations loaded")
    
    return melted

# ============================================
# 2. Load All Indicators
# ============================================

print("\n" + "=" * 60)
print("📂 LOADING DATA FROM CSV FILES")
print("=" * 60)

# Define file mappings (update these paths to match your actual file locations)
# If the files are in the same folder as your notebook:
data_files = {
    'API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_4522': 'Inflation_Rate',
    'API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_3530': 'GDP_Growth',
    'API_NY.GDP.PCAP.KD_DS2_en_csv_v2_3781': 'GDP_Per_Capita',
    'API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_3359': 'Unemployment'
}

# If files are in a subfolder called 'data' or 'raw', adjust the path:
# data_files = {
#     'data/API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_4522': 'Inflation_Rate',
#     ...
# }

loaded_data = {}

for filename, indicator_name in data_files.items():
    try:
        # If the file is in the current directory
        filepath = f"{filename}.csv"  # or filename if no .csv extension
        
        # If files are in a 'data' folder:
        # filepath = f"data/{filename}.csv"
        
        df = load_wb_csv(filepath, indicator_name)
        loaded_data[indicator_name] = df
        
    except FileNotFoundError:
        print(f"\n❌ File not found: {filepath}")
        print(f"   Please check the file name and location")
        print(f"   Expected file: {filename}.csv")
        
# Check if all files loaded
if len(loaded_data) < 4:
    print("\n⚠️ Not all files loaded. Please check file names and paths.")
    print("   Current directory contents:")
    import os
    for file in os.listdir('.'):
        if file.endswith('.csv'):
            print(f"   - {file}")

# ============================================
# 3. Merge All Indicators
# ============================================

print("\n" + "=" * 60)
print("🔗 MERGING INDICATORS")
print("=" * 60)

# Start with inflation
if 'Inflation_Rate' in loaded_data:
    merged = loaded_data['Inflation_Rate'].copy()
    
    # Merge other indicators
    for indicator in ['GDP_Growth', 'GDP_Per_Capita', 'Unemployment']:
        if indicator in loaded_data:
            print(f"   Merging {indicator}...")
            merged = merged.merge(
                loaded_data[indicator][['economy', 'Country', 'Year', indicator]],
                on=['economy', 'Country', 'Year'],
                how='left'
            )
    
    print(f"✅ Merged dataset: {len(merged)} observations")
    print(f"   Columns: {list(merged.columns)}")
else:
    print("❌ No data loaded. Cannot proceed.")
    exit()

# ============================================
# 4. Add Region Information
# ============================================

print("\n" + "=" * 60)
print("🌍 ADDING REGION INFORMATION")
print("=" * 60)

# Basic region mapping (add more as needed)
region_mapping = {
    # North America
    'USA': 'North America', 'CAN': 'North America', 'MEX': 'North America',
    
    # South America
    'BRA': 'South America', 'ARG': 'South America', 'CHL': 'South America',
    'COL': 'South America', 'PER': 'South America', 'VEN': 'South America',
    'ECU': 'South America', 'BOL': 'South America', 'PRY': 'South America',
    'URY': 'South America',
    
    # Europe
    'GBR': 'Europe', 'DEU': 'Europe', 'FRA': 'Europe', 'ITA': 'Europe',
    'ESP': 'Europe', 'NLD': 'Europe', 'BEL': 'Europe', 'CHE': 'Europe',
    'SWE': 'Europe', 'NOR': 'Europe', 'DNK': 'Europe', 'FIN': 'Europe',
    'POL': 'Europe', 'TUR': 'Europe', 'RUS': 'Europe', 'PRT': 'Europe',
    'AUT': 'Europe', 'GRC': 'Europe', 'IRL': 'Europe', 'ISR': 'Europe',
    
    # Asia
    'CHN': 'Asia', 'JPN': 'Asia', 'KOR': 'Asia', 'IND': 'Asia',
    'IDN': 'Asia', 'SAU': 'Asia', 'IRN': 'Asia', 'PAK': 'Asia',
    'BGD': 'Asia', 'PHL': 'Asia', 'VNM': 'Asia', 'THA': 'Asia',
    'MYS': 'Asia', 'SGP': 'Asia', 'LKA': 'Asia', 'NPL': 'Asia',
    
    # Africa
    'ZAF': 'Africa', 'NGA': 'Africa', 'EGY': 'Africa', 'KEN': 'Africa',
    'ETH': 'Africa', 'GHA': 'Africa', 'MAR': 'Africa', 'TUN': 'Africa',
    'DZA': 'Africa', 'AGO': 'Africa', 'CMR': 'Africa', 'CIV': 'Africa',
    
    # Oceania
    'AUS': 'Oceania', 'NZL': 'Oceania',
    
    # Middle East
    'ARE': 'Middle East', 'QAT': 'Middle East', 'KWT': 'Middle East',
    'BHR': 'Middle East', 'OMN': 'Middle East', 'IRQ': 'Middle East',
}

income_mapping = {
    # High income
    'USA': 'High', 'CAN': 'High', 'GBR': 'High', 'DEU': 'High', 'FRA': 'High',
    'ITA': 'High', 'ESP': 'High', 'NLD': 'High', 'BEL': 'High', 'CHE': 'High',
    'SWE': 'High', 'NOR': 'High', 'DNK': 'High', 'FIN': 'High', 'AUS': 'High',
    'NZL': 'High', 'JPN': 'High', 'KOR': 'High', 'SAU': 'High', 'SGP': 'High',
    'ISR': 'High', 'ARE': 'High', 'QAT': 'High', 'KWT': 'High', 'BHR': 'High',
    'OMN': 'High', 'IRL': 'High', 'PRT': 'High', 'GRC': 'High', 'AUT': 'High',
    'CHL': 'High', 'URY': 'High',
    
    # Upper middle
    'MEX': 'Upper Middle', 'BRA': 'Upper Middle', 'ARG': 'Upper Middle',
    'CHN': 'Upper Middle', 'TUR': 'Upper Middle', 'RUS': 'Upper Middle',
    'ZAF': 'Upper Middle', 'MYS': 'Upper Middle', 'THA': 'Upper Middle',
    'COL': 'Upper Middle', 'PER': 'Upper Middle', 'ECU': 'Upper Middle',
    'BOL': 'Upper Middle', 'PRY': 'Upper Middle', 'DZA': 'Upper Middle',
    'TUN': 'Upper Middle',
    
    # Lower middle
    'IND': 'Lower Middle', 'IDN': 'Lower Middle', 'PHL': 'Lower Middle',
    'VNM': 'Lower Middle', 'MAR': 'Lower Middle', 'EGY': 'Lower Middle',
    'NGA': 'Lower Middle', 'KEN': 'Lower Middle', 'PAK': 'Lower Middle',
    'BGD': 'Lower Middle', 'IRN': 'Lower Middle', 'LKA': 'Lower Middle',
    'NPL': 'Lower Middle', 'CMR': 'Lower Middle', 'CIV': 'Lower Middle',
    'AGO': 'Lower Middle',
    
    # Low
    'ETH': 'Low', 'MWI': 'Low', 'UGA': 'Low', 'MOZ': 'Low',
    'TZA': 'Low', 'RWA': 'Low', 'NGA': 'Low', 'KEN': 'Low',
}

# Add region info
merged['Region'] = merged['economy'].map(region_mapping).fillna('Other')
merged['IncomeLevel'] = merged['economy'].map(income_mapping).fillna('Unknown')

# Show what we added
print(f"✅ Added Region and IncomeLevel")
print(f"   Regions: {merged['Region'].unique().tolist()}")
print(f"   Income Levels: {merged['IncomeLevel'].unique().tolist()}")

# ============================================
# 5. Add Derived Fields
# ============================================

print("\n" + "=" * 60)
print("🧮 ADDING DERIVED FIELDS")
print("=" * 60)

# Add decade
merged['Decade'] = (merged['Year'] // 10) * 10

# Inflation categories
def inflation_category(rate):
    if pd.isna(rate):
        return 'Unknown'
    elif rate < 0:
        return 'Deflation'
    elif rate < 2:
        return 'Low'
    elif rate < 5:
        return 'Moderate'
    elif rate < 10:
        return 'High'
    else:
        return 'Very High'

merged['Inflation_Category'] = merged['Inflation_Rate'].apply(inflation_category)

# High inflation flag
merged['High_Inflation'] = merged['Inflation_Rate'] > 10

print(f"✅ Added derived fields")

# ============================================
# 6. Save Dataset
# ============================================

print("\n" + "=" * 60)
print("💾 SAVING DATASET")
print("=" * 60)

# Save
output_file = 'inflation_clean.csv'
merged.to_csv(output_file, index=False)
print(f"✅ Data saved to '{output_file}'")

# ============================================
# 7. Summary Statistics
# ============================================

print("\n" + "=" * 60)
print("📊 DATASET SUMMARY")
print("=" * 60)

print(f"\n📈 Overview:")
print(f"   Countries: {merged['Country'].nunique()}")
print(f"   Years: {merged['Year'].min()} - {merged['Year'].max()}")
print(f"   Total Observations: {len(merged)}")
print(f"   Inflation Range: {merged['Inflation_Rate'].min():.1f}% to {merged['Inflation_Rate'].max():.1f}%")

print(f"\n🌍 Top 10 Average Inflation by Country (1990-2023):")
top_inflation = merged.groupby('Country')['Inflation_Rate'].mean().sort_values(ascending=False).head(10)
for country, rate in top_inflation.items():
    print(f"   {country}: {rate:.1f}%")

print(f"\n🏷️ Data Completeness:")
for col in ['Inflation_Rate', 'GDP_Growth', 'GDP_Per_Capita', 'Unemployment']:
    if col in merged.columns:
        non_na = merged[col].notna().sum()
        pct = (non_na / len(merged)) * 100
        print(f"   {col}: {non_na:,} / {len(merged):,} ({pct:.1f}%)")

print(f"\n🌍 Regions Distribution:")
print(merged['Region'].value_counts().to_string())

print("\n✅ Data processing complete!")

📊 GLOBAL INFLATION ANALYSIS - DATA PROCESSING

📂 LOADING DATA FROM CSV FILES

📄 Loading Inflation_Rate...
   Shape: 265 rows, 71 columns
   Columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960']...
   ✅ 7418 observations loaded

📄 Loading GDP_Growth...
   Shape: 265 rows, 71 columns
   Columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960']...
   ✅ 8507 observations loaded

📄 Loading GDP_Per_Capita...
   Shape: 265 rows, 71 columns
   Columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960']...
   ✅ 8503 observations loaded

📄 Loading Unemployment...
   Shape: 265 rows, 71 columns
   Columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960']...
   ✅ 7718 observations loaded

🔗 MERGING INDICATORS
   Merging GDP_Growth...
   Merging GDP_Per_Capita...
   Merging Unemployment...
✅ Merged dataset: 7418 observations
   Columns: ['Country', 'economy', 'Indicator Name', 'Indicato